# Libraries

In [2]:
import json
import math 
import sys
from pathlib import Path 
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.ocr_loader import load_ocr

# Load annotations

In [3]:
ANNOTATION_DIR = ROOT / "data/raw/annotations"
OCR_DIR = ROOT / "data/raw/ocr"

annotation_files = {
    "train": "train_v1.0_withQT.json",
    "val": "val_v1.0_withQT.json",
    "test": "test_v1.0.json",
}

records_by_split = {}
for split, filename in annotation_files.items():
    path = ANNOTATION_DIR / filename
    with path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    records_by_split[split] = payload["data"]

print("Project folder:", ROOT)

for split, records in records_by_split.items():
    print(split, "questions:", len(records))

Project folder: C:\Users\firma\Documents\LOCKIN\SP-DOCVQA
train questions: 39463
val questions: 5349
test questions: 5188


# Dataset summary

In [7]:
def get_answers(record):
    answers = record.get("answers") or record.get("answer") or []
    if isinstance(answers, str):
        answers = [answers]
    return [str(answer).strip() for answer in answers if str(answer).strip()]

dataset_rows = []
question_per_document_rows = []
for split, records in records_by_split.items():
    document_names = [Path(record["image"]).stem.lower() for record in records]
    document_counts = pd.Series(document_names).value_counts()
    questions_with_answers = sum(bool(get_answers(record)) for record in records)

    dataset_rows.append({
        "split": split,
        "documents": len(document_counts),
        "questions": len(records),
        "questions_with_answers": questions_with_answers,
        "reference_answers": "Available" if questions_with_answers > 0 else "Not Available",
        "questions_per_document": round(len(records) / len(document_counts), 2)
    })

    for document, question_count in document_counts.items():
        question_per_document_rows.append({
            "split": split,
            "document": document,
            "question_count": question_count
        })

dataset_summary = pd.DataFrame(dataset_rows)
question_per_document_df = pd.DataFrame(question_per_document_rows)
display(dataset_summary)

,split,documents,questions,questions_with_answers,reference_answers,questions_per_document
0,train,10194,39463,39463,Available,3.87
1,val,1286,5349,5349,Available,4.16
2,test,1287,5188,0,Not Available,4.03


## Documents with multiple questions

In [11]:
multi_question_summary = question_per_document_df.groupby("split").agg(
    documents = ("document", "count"),
    mean_questions = ("question_count", "mean"),
    median_questions = ("question_count", "median"),
    max_questions = ("question_count", "max"),
    documents_with_multiple_questions=("question_count", lambda values: (values > 1).sum())
).reset_index()

multi_question_summary["multiple_question_percent"] = (
    multi_question_summary["documents_with_multiple_questions"] / multi_question_summary["documents"] * 100
)

display(multi_question_summary.round(2))

,split,documents,mean_questions,median_questions,max_questions,documents_with_multiple_questions,multiple_question_percent
0,test,1287,4.03,4.0,31,1140,88.58
1,train,10194,3.87,3.0,12,8395,82.35
2,val,1286,4.16,4.0,13,1146,89.11


## Answer length

In [12]:
answer_rows = []
for split, records in records_by_split.items():
    for record in records:
        for answer in get_answers(record):
            answer_rows.append({
                "split": split,
                "answer": answer,
                "word_count": len(answer.split())
            })

answer_df = pd.DataFrame(answer_rows)
answer_df["length_group"] = pd.cut(
    answer_df["word_count"],
    bins=[0, 1, 3, 5, 10, 15, 20, float("inf")],
    labels=[
        "1 word",
        "2-3 words",
        "4-5 words",
        "6-10 words",
        "11-15 words",
        "16-20 words",
        "over 20 words"
    ]
)

answer_length_summary = answer_df.groupby("split").agg(
        answers=("answer", "count"),
        mean_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        max_words=("word_count", "max"),
        answers_up_to_10_words=("word_count", lambda values: (values <= 10).sum())
    ).reset_index()

answer_length_summary["up_to_10_percent"] = (
    answer_length_summary["answers_up_to_10_words"] / answer_length_summary["answers"] * 100
)

answer_group_summary = (
    answer_df.groupby(["split", "length_group"], observed=True)
    .size().reset_index(name="answers")
)

answer_group_summary["percent"] = (
    answer_group_summary["answers"] / answer_group_summary
    .groupby("split")["answers"].transform("sum") * 100
)

display(answer_length_summary.round(2))
display(answer_group_summary.round(2))

,split,answers,mean_words,median_words,max_words,answers_up_to_10_words,up_to_10_percent
0,train,56259,2.35,2.0,30,55948,99.45
1,val,8209,2.44,2.0,24,8134,99.09


,split,length_group,answers,percent
0,train,1 word,23977,42.62
1,train,2-3 words,22554,40.09
2,train,4-5 words,6818,12.12
3,train,6-10 words,2599,4.62
4,train,11-15 words,258,0.46
5,train,16-20 words,42,0.07
6,train,over 20 words,11,0.02
7,val,1 word,3523,42.92
8,val,2-3 words,3145,38.31
9,val,4-5 words,971,11.83


## OCR words and low labels

In [ ]:
ocr_index = {path.stem.lower(): path for path in OCR_DIR.rglob("*.json")}

ocr_rows = []
for split, records in records_by_split.items():
    document_names = {Path(record["image"]).stem.lower() for record in records}
    for document in sorted(document_names):
        ocr_path = ocr_index.get(document)
        if ocr_path is None:
            ocr_rows.append({
                "split": split,
                "document": document,
                "ocr_found": False,
                "word_count": None,
                "low_count": None
            })
            continue

        try:
            words = load_ocr(ocr_path)["words"]
        except ValueError:
            words = []

        low_count = sum(str(word.get("confidence_label") or "").strip().lower() == "low" for word in words)

        ocr_rows.append({
            "split": split,
            "document": document,
            "ocr_found": True,
            "word_count": len(words),
            "low_count": low_count
        })

ocr_df = pd.DataFrame(ocr_rows)
ocr_df["low_percent_per_document"] = (
    ocr_df["low_count"] / ocr_df["word_count"].where(ocr_df["word_count"] > 0) * 100
)

ocr_summary = (
    ocr_df.groupby("split").agg(
        documents=("document", "count"),
        ocr_files_found=("ocr_found", "sum"),
        total_ocr_words=("word_count", "sum"),
        mean_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        p90_words=("word_count", lambda values: values.quantile(0.9)),
        max_words=("word_count", "max"),
        total_low_words=("low_count", "sum"),
        documents_with_low=("low_count", lambda values: (values > 0).sum())
    ).reset_index()
)

ocr_summary["low_word_percent"] = (
    ocr_summary["total_low_words"] / ocr_summary["total_ocr_words"] * 100
)

display(ocr_summary.round(2))

,split,documents,ocr_files_found,total_ocr_words,mean_words,median_words,p90_words,max_words,total_low_words,documents_with_low,low_word_percent
0,test,1287,1287,252876,196.48,158.0,405.2,1520,22731,1212,8.99
1,train,10194,10194,1824105,178.94,146.0,349.0,1844,179586,9684,9.85
2,val,1286,1286,256215,199.23,159.5,408.5,1176,22765,1225,8.89


## Window Count Estimate

In [ ]:
window_settings = [(100, 50), (160, 60), (200, 100)]

window_rows = []
for max_words, stride in window_settings:
    step = max_words - stride

    for row in ocr_df.itertuples():
        word_count = row.word_count
        if pd.isna(word_count) or word_count == 0:
            window_count = 0
        elif word_count <= max_words:
            window_count = 1
        else:
            window_count = math.ceil((word_count - max_words) / step) + 1

        window_rows.append({
            "split": row.split,
            "max_words": max_words,
            "stride": stride,
            "estimated_windows": window_count
        })

window_estimate_df = pd.DataFrame(window_rows)

window_summary = window_estimate_df.groupby(["split", "max_words", "stride"]).agg(
        estimated_total_windows=("estimated_windows", "sum"),
        mean_windows_per_document=("estimated_windows", "mean"),
        documents_with_multiple_windows=("estimated_windows", lambda values: (values > 1).sum()
    )
).reset_index()

display(window_summary.round(2))

,split,max_words,stride,estimated_total_windows,mean_windows_per_document,documents_with_multiple_windows
0,test,100,50,4577,3.56,866
1,test,160,60,2602,2.02,632
2,test,200,100,2313,1.80,483
3,train,100,50,32499,3.19,6795
4,train,160,60,18720,1.84,4595
5,train,200,100,16526,1.62,3382
6,val,100,50,4625,3.60,880
7,val,160,60,2638,2.05,640
8,val,200,100,2322,1.81,494


# Save the tables

In [16]:
OUTPUT_DIR = ROOT / "outputs/eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "dataset.csv": dataset_summary,
    "questions_per_document.csv": multi_question_summary,
    "answer_lengths.csv": answer_length_summary,
    "answer_groups.csv": answer_group_summary,
    "ocr.csv": ocr_summary,
    "windows.csv": window_summary,
}

for filename, table in tables.items():
    table.to_csv(
        OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig",
    )

print("Saved to:", OUTPUT_DIR)

Saved to: C:\Users\firma\Documents\LOCKIN\SP-DOCVQA\outputs\eda
